# PART 2: Training


In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from collections import Counter

# Set random seed for reproducibility
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed()



### 1. Define Augmentations




In [2]:
# Justification: Resize to 224 for standard models.
# HorizontalFlip and Rotation account for different camera angles in the field.
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])



### 2. Load [Datasets](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.Flowers102.html#torchvision.datasets.Flowers102)

In [3]:
train_set= datasets.Flowers102(root='data', split='test', download=True, transform=train_transforms)
test_set = datasets.Flowers102(root='data', split='train', download=True, transform=test_val_transforms)
val_set = datasets.Flowers102(root='data', split='val', download=True, transform=test_val_transforms)


100%|██████████| 345M/345M [00:13<00:00, 25.6MB/s]
100%|██████████| 502/502 [00:00<00:00, 2.09MB/s]
100%|██████████| 15.0k/15.0k [00:00<00:00, 44.4MB/s]


### 3. DataLoaders

In [4]:
from torch.utils.data import WeightedRandomSampler

# Calculate weights for the Training set to handle imbalance
targets = train_set._labels
class_sample_count = np.array([len(np.where(targets == t)[0]) for t in np.unique(targets)])
weight = 1. / class_sample_count
samples_weight = np.array([weight[t] for t in targets])
sampler = WeightedRandomSampler(torch.from_numpy(samples_weight).double(), len(samples_weight))

# Final Loaders
train_loader = DataLoader(train_set, batch_size=32, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=2)

print("✅ Data Pipeline Complete: Part 1 Finished.")

✅ Data Pipeline Complete: Part 1 Finished.



### 4. Model Setup (EfficientNet-B0)
* For fine-grained classification (distinguishing between specific flower species), standard ResNets are good, but EfficientNet or Vision Transformers (ViT) are often better because they capture subtle feature hierarchies more effectively.

* Faster training on Colab's free GPU and less risk of overfitting on a small dataset (102 classes, ~80 images each)

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
import time
import copy

def get_model(num_classes=102, feature_extract=True):
    # Load pre-trained EfficientNet
    # "weights='DEFAULT'" loads the best available ImageNet weights
    model = models.efficientnet_b0(weights='DEFAULT')

    # Freeze the backbone weights if feature_extract is True
    # This prevents destroying learned features in the first few epochs
    if feature_extract:
        for param in model.parameters():
            param.requires_grad = False

    # Replace the classifier head
    # EfficientNet's classifier is a Sequential block; we access the last Linear layer
    # in_features for B0 is usually 1280
    num_ftrs = model.classifier[1].in_features

    # New Head: Unfrozen by default, so it will learn from scratch
    model.classifier[1] = nn.Linear(num_ftrs, num_classes)

    return model


### 5. Early Stopping Utility

In [6]:

class EarlyStopping:
    """Stops training if validation loss doesn't improve after a given patience."""
    def __init__(self, patience=5, delta=0, path='best_model.pth'):
        self.patience = patience
        self.delta = delta
        self.path = path
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        """Saves model when validation loss decreases."""
        torch.save(model.state_dict(), self.path)
        print(f'Validation loss decreased. Saving model to {self.path}...')


### 6. Training Loop Structure

In [7]:

def train_model(model, dataloaders, criterion, optimizer, num_epochs=25, patience=5):
    since = time.time()

    # Initialize Early Stopping
    early_stopping = EarlyStopping(patience=patience, path='flower_model_best.pth')

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    # Lists to store metrics for plotting later
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # Statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects.double() / len(dataloaders[phase].dataset)

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Save history
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.item())
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.item())

                # Check Early Stopping
                early_stopping(epoch_loss, model)

        if early_stopping.early_stop:
            print("Early stopping triggered.")
            break

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')

    # Load the best model weights before returning
    model.load_state_dict(torch.load('flower_model_best.pth'))
    return model, history

print("✅ Model Architecture & Training Infrastructure Ready")

✅ Model Architecture & Training Infrastructure Ready


### 7. Execution


In [8]:
import torch
print(f"Is GPU available? {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("❌ Running on CPU. See instructions below to switch.")

Is GPU available? True
Device Name: Tesla T4


In [9]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# Initialize Model
model_ft = get_model(num_classes=102, feature_extract=False).to(device) # feature_extract=False means fine-tune whole model

# Define Loss and Optimizer
criterion = nn.CrossEntropyLoss()
# Lower learning rate for fine-tuning
optimizer_ft = optim.AdamW(model_ft.parameters(), lr=1e-4, weight_decay=1e-4)

# Loaders dictionary (assuming you created these in Part 1)
dataloaders_dict = {'train': train_loader, 'val': val_loader}

# TRAIN!
model_ft, history = train_model(model_ft, dataloaders_dict, criterion, optimizer_ft, num_epochs=20, patience=5)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 158MB/s]


Epoch 1/20
----------
train Loss: 3.4983 Acc: 0.4352
val Loss: 1.9824 Acc: 0.7382
Validation loss decreased. Saving model to flower_model_best.pth...

Epoch 2/20
----------
train Loss: 1.2576 Acc: 0.8411
val Loss: 0.6636 Acc: 0.9059
Validation loss decreased. Saving model to flower_model_best.pth...

Epoch 3/20
----------
train Loss: 0.4628 Acc: 0.9467
val Loss: 0.3206 Acc: 0.9569
Validation loss decreased. Saving model to flower_model_best.pth...

Epoch 4/20
----------
train Loss: 0.2285 Acc: 0.9712
val Loss: 0.2424 Acc: 0.9578
Validation loss decreased. Saving model to flower_model_best.pth...

Epoch 5/20
----------
train Loss: 0.1529 Acc: 0.9806
val Loss: 0.1775 Acc: 0.9657
Validation loss decreased. Saving model to flower_model_best.pth...

Epoch 6/20
----------
train Loss: 0.1022 Acc: 0.9844
val Loss: 0.1500 Acc: 0.9657
Validation loss decreased. Saving model to flower_model_best.pth...

Epoch 7/20
----------
train Loss: 0.0671 Acc: 0.9911
val Loss: 0.1411 Acc: 0.9696
Validation l

In [10]:
%mv flower_model_best.pth /content/drive/MyDrive

mv: cannot move 'flower_model_best.pth' to '/content/drive/MyDrive': No such file or directory
